Created with the help of chat.ai using model Anthropic Claude Sonnet 5

In [1]:
import sqlite3
import re

In [48]:
def parse_msp_stream(filepath):
    meta, peaks, in_peaks = {}, [], False

    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.strip() == "":
                if meta:
                    yield {"meta": meta, "peaks": peaks}
                meta, peaks, in_peaks = {}, [], False
                continue

            if re.match(r"^Num\s*Peaks", line, re.IGNORECASE):
                in_peaks = True
                continue

            if in_peaks:
                parts = re.split(r"[\s,;]+", line.strip())
                if len(parts) >= 2:
                    try:
                        peaks.append((float(parts[0]), float(parts[1])))
                    except ValueError:
                        pass
            elif ":" in line:
                key, _, val = line.partition(":")
                meta[key.strip()] = val.strip()

        if meta:
            yield {"meta": meta, "peaks": peaks}


In [49]:
def split_mona_by_ionmode(msp_file, out_pos="gnps_pos.sqlite", out_neg="gnps_neg.sqlite",
                           batch_size=1000, DB="GNPS"):

    schema = """CREATE TABLE IF NOT EXISTS spectra (
        compound_id TEXT, name TEXT, inchikey TEXT, formula TEXT,
        adduct TEXT, precursor_mz REAL, polarity TEXT,
        collision_energy TEXT, mz TEXT, intensity TEXT, library_source TEXT)"""
    insert_sql = """INSERT INTO spectra VALUES (?,?,?,?,?,?,?,?,?,?,?)"""

    con_pos, con_neg = sqlite3.connect(out_pos), sqlite3.connect(out_neg)
    con_pos.execute(schema); con_neg.execute(schema)

    batch_pos, batch_neg, n_pos, n_neg = [], [], 0, 0

    for rec in parse_msp_stream(msp_file):


        if DB == "MoNA":
            meta, peaks = rec["meta"], rec["peaks"]
            if not peaks:
                continue

            mode_raw = meta.get("Ion_mode", meta.get("Ionization mode", "")).upper()
            mode = "P" if mode_raw.startswith("P") else ("N" if mode_raw.startswith("N") else None)
            if mode is None:
                continue

            if "PrecursorMZ" in meta:
                if len(meta["PrecursorMZ"]) > 1:
                    meta.pop("PrecursorMZ")

            row = (
                meta.get("DB#"), meta.get("Name"), meta.get("InChIKey"), meta.get("Formula"),
                meta.get("Precursor_type"),
                float(meta["PrecursorMZ"]) if "PrecursorMZ" in meta else None,
                mode, meta.get("Collision_energy"),
                ";".join(str(p[0]) for p in peaks), ";".join(str(p[1]) for p in peaks),
                "MoNA"
            )

            if mode == "P":
                batch_pos.append(row); n_pos += 1
            else:
                batch_neg.append(row); n_neg += 1

        elif DB == "GNPS":
            meta, peaks = rec["meta"], rec["peaks"]
            if not peaks:
                continue

            mode_raw = meta.get("IONMODE", meta.get("Ionization mode", "")).upper()
            mode = "P" if mode_raw.startswith("P") else ("N" if mode_raw.startswith("N") else None)
            if mode is None:
                continue

            if "PRECURSORMZ" in meta:
                if len(meta["PRECURSORMZ"]) > 1:
                    meta.pop("PRECURSORMZ")

            meta["DB#"] = meta["Comment"].split(";")[0].split("=")[1]

            row = (
                meta.get("DB#"), meta.get("NAME"), meta.get("INCHIKEY"), meta.get("FORMULA"),
                meta.get("PRECURSORTYPE"),
                float(meta["PRECURSORMZ"]) if "PRECURSORMZ" in meta else None,
                mode, meta.get("COLLISIONENERGY"),
                ";".join(str(p[0]) for p in peaks), ";".join(str(p[1]) for p in peaks),
                "GNPS"
            )

            if mode == "P":
                batch_pos.append(row); n_pos += 1
            else:
                batch_neg.append(row); n_neg += 1

        if len(batch_pos) >= batch_size:
            con_pos.executemany(insert_sql, batch_pos); con_pos.commit(); batch_pos = []
        if len(batch_neg) >= batch_size:
            con_neg.executemany(insert_sql, batch_neg); con_neg.commit(); batch_neg = []

    if batch_pos: con_pos.executemany(insert_sql, batch_pos); con_pos.commit()
    if batch_neg: con_neg.executemany(insert_sql, batch_neg); con_neg.commit()
    con_pos.close(); con_neg.close()
    print(f"Done. Positive: {n_pos}, Negative: {n_neg}")

In [50]:
split_mona_by_ionmode("ALL_GNPS_NO_PROPOGATED.msp")

Done. Positive: 713370, Negative: 240680
